# Preparacion para clustering

Este notebook prepara una matriz numerica inicial para clustering usando variables del PDF. Tambien genera una vista PCA de dos componentes para revisar si hay estructura antes de probar algoritmos como K-Means, Gaussian Mixture, DBSCAN o HDBSCAN.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from feature_config import CLUSTERING_FEATURE_SETS, LOG_CANDIDATE_COLUMNS, RADIUS_BINS, RADIUS_LABELS

pio.renderers.default = 'notebook_connected'
csv_path = sorted(PROJECT_ROOT.glob('PSCompPars_*.csv'))[-1]
df = pd.read_csv(csv_path, comment='#', low_memory=False)
display(Markdown(f'**Dataset:** `{csv_path.name}` con `{df.shape[0]}` filas y `{df.shape[1]}` columnas.'))

**Dataset:** `PSCompPars_2026.04.25_14.43.08.csv` con `6273` filas y `320` columnas.

## Seleccion de features

El set inicial recomendado conserva buena cobertura y evita duplicar unidades (`pl_rade` vs `pl_radj`, `pl_bmasse` vs `pl_bmassj`).

In [2]:
feature_set_name = 'pdf_core_with_mass_density'
features = CLUSTERING_FEATURE_SETS[feature_set_name]
features = [col for col in features if col in df.columns]

coverage = df[features].notna().all(axis=1).mean() * 100
features_inline = '`, `'.join(features)
message = (
    f"**Feature set:** `{feature_set_name}`  \n"
    f"**Features:** `{features_inline}`  \n"
    f"**Filas completas:** `{coverage:.2f}%`"
)
display(Markdown(message))

df[features].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

**Feature set:** `pdf_core_with_mass_density`  
**Features:** `pl_rade`, `pl_bmasse`, `pl_dens`, `pl_orbper`, `pl_orbsmax`, `pl_orbeccen`, `st_teff`, `st_met`, `sy_pnum`  
**Filas completas:** `77.99%`

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
pl_rade,6212.0,5.797036,5.400699e+00,0.309800,0.750449,1.071650,1.8400,2.840000,11.890682,14.300000,18.260806,8.720587e+01
pl_bmasse,6231.0,400.191279,1.135000e+03,0.020000,0.383100,1.330000,4.2700,9.190000,181.163100,2511.162244,6477.290500,9.534852e+03
pl_dens,6123.0,4.854600,3.423458e+01,0.005100,0.160000,0.330100,1.3000,2.530000,4.540000,9.740000,27.456000,2.000000e+03
pl_orbper,5929.0,72191.071478,5.223222e+06,0.090706,0.647353,1.555385,4.3050,10.681572,38.021000,1527.000000,13689.235949,4.020000e+08
pl_orbsmax,5834.0,15.723426,3.490438e+02,0.004400,0.013233,0.024800,0.0523,0.102085,0.307750,4.064900,67.670000,1.900000e+04
pl_orbeccen,5210.0,0.079067,1.527213e-01,0.000000,0.000000,0.000000,0.0000,0.000000,0.090000,0.410000,0.730000,9.500000e-01
st_teff,5959.0,5392.155798,1.735933e+03,415.000000,3099.580000,3514.900000,4895.9850,5542.000000,5894.500000,6340.000000,7268.260000,5.700000e+04
st_met,5609.0,0.016194,1.882722e-01,-1.000000,-0.540000,-0.306900,-0.0800,0.020000,0.130000,0.320000,0.410000,7.900000e-01
sy_pnum,6273.0,1.759126,1.149467e+00,1.000000,1.000000,1.000000,1.0000,1.000000,2.000000,4.000000,6.000000,8.000000e+00


## Transformaciones

Las variables positivas y muy sesgadas se transforman con `log10`. Despues se imputan nulos con mediana y se escala todo con `StandardScaler`.

In [3]:
log_features = [col for col in features if col in LOG_CANDIDATE_COLUMNS]
linear_features = [col for col in features if col not in log_features]

def safe_log10(values):
    frame = pd.DataFrame(values, columns=log_features).copy()
    frame = frame.where(frame > 0)
    return np.log10(frame)

preprocess = ColumnTransformer(
    transformers=[
        ('log', Pipeline([
            ('log10', FunctionTransformer(safe_log10, validate=False, feature_names_out='one-to-one')),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), log_features),
        ('linear', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), linear_features),
    ],
    remainder='drop',
)

X = preprocess.fit_transform(df[features])
transformed_feature_names = [f'log10_{col}' for col in log_features] + linear_features
X_df = pd.DataFrame(X, columns=transformed_feature_names, index=df.index)
X_df.head()

,log10_pl_rade,log10_pl_bmasse,log10_pl_dens,log10_pl_orbper,log10_pl_orbsmax,pl_orbeccen,st_teff,st_met,sy_pnum
0,1.268626,2.216973,1.804981,1.434475,1.190421,1.211097,-0.310653,-1.553766,-0.660468
1,1.277454,2.197071,1.731662,1.663429,1.346216,0.100719,-0.701291,-0.205581,-0.660468
2,1.345596,1.607174,0.193014,1.166282,0.940914,-0.461498,-0.302379,-1.272894,-0.660468
3,1.294896,1.987645,1.205624,2.264955,1.714588,2.126807,-0.036438,2.322264,0.209569
4,1.378121,1.319566,-0.556616,1.876722,1.394811,4.317343,0.207046,0.243813,-0.660468


In [4]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_df)
pca_df = pd.DataFrame(coords, columns=['PC1', 'PC2'])
pca_df['pl_name'] = df['pl_name'].values
pca_df['hostname'] = df['hostname'].values
pca_df['discoverymethod'] = df['discoverymethod'].values
pca_df['pl_rade'] = df['pl_rade'].values
pca_df['radius_class'] = pd.cut(df['pl_rade'], bins=RADIUS_BINS, labels=RADIUS_LABELS).astype(str)

display(Markdown(
    f'**Varianza explicada:** PC1 `{pca.explained_variance_ratio_[0]:.2%}`, '
    f'PC2 `{pca.explained_variance_ratio_[1]:.2%}`'
))
pca_df.head()

**Varianza explicada:** PC1 `35.14%`, PC2 `16.70%`

,PC1,PC2,pl_name,hostname,discoverymethod,pl_rade,radius_class
0,2.794160,2.016134,11 Com b,11 Com,Radial Velocity,12.2,Gigante gaseoso (>10)
1,2.703407,1.461599,11 UMi b,11 UMi,Radial Velocity,12.3,Gigante gaseoso (>10)
2,2.064024,0.527213,14 And b,14 And,Radial Velocity,13.1,Gigante gaseoso (>10)
3,3.980898,1.278946,14 Her b,14 Her,Radial Velocity,12.5,Gigante gaseoso (>10)
4,4.310564,0.951174,16 Cyg B b,16 Cyg B,Radial Velocity,13.5,Gigante gaseoso (>10)


In [5]:
fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='discoverymethod',
    symbol='radius_class',
    hover_name='pl_name',
    hover_data=['hostname', 'pl_rade'],
    title='PCA exploratorio de variables preparadas para clustering',
    height=650,
)
fig.show()

## Matriz lista para modelado

La tabla `X_df` queda imputada, transformada y escalada. Es la entrada natural para probar clustering. Mantener `df[['pl_name', 'hostname', 'discoverymethod']]` separado ayuda a interpretar los grupos sin meter identificadores al modelo.

In [6]:
model_ready = pd.concat(
    [df[['pl_name', 'hostname', 'discoverymethod']].reset_index(drop=True), X_df.reset_index(drop=True)],
    axis=1,
)
model_ready.head()

,pl_name,hostname,discoverymethod,log10_pl_rade,log10_pl_bmasse,log10_pl_dens,log10_pl_orbper,log10_pl_orbsmax,pl_orbeccen,st_teff,st_met,sy_pnum
0,11 Com b,11 Com,Radial Velocity,1.268626,2.216973,1.804981,1.434475,1.190421,1.211097,-0.310653,-1.553766,-0.660468
1,11 UMi b,11 UMi,Radial Velocity,1.277454,2.197071,1.731662,1.663429,1.346216,0.100719,-0.701291,-0.205581,-0.660468
2,14 And b,14 And,Radial Velocity,1.345596,1.607174,0.193014,1.166282,0.940914,-0.461498,-0.302379,-1.272894,-0.660468
3,14 Her b,14 Her,Radial Velocity,1.294896,1.987645,1.205624,2.264955,1.714588,2.126807,-0.036438,2.322264,0.209569
4,16 Cyg B b,16 Cyg B,Radial Velocity,1.378121,1.319566,-0.556616,1.876722,1.394811,4.317343,0.207046,0.243813,-0.660468
